# EzLab — Colab LoRA → GGUF

Fine-tune **Qwen2.5-1.5B-Instruct** on the CS lab-report dataset, then export `labpulse-1.5b-q4_k_m.gguf` for llama.cpp.

## Setup (do this once)
1. **Runtime → Change runtime type → GPU → T4 → Save**
2. On your Mac, keep this file ready: `Labpulse/data/colab_dataset.zip` (or `train.jsonl` + `valid.jsonl`)
3. **Runtime → Run all**
4. When the upload cell runs, choose that zip (or both jsonl files)
5. After training (~30–90 min), download the GGUF into `Labpulse/models/`

```text
Labpulse/models/labpulse-1.5b-q4_k_m.gguf
```

**Note:** This notebook does **not** clone GitHub (your repo is private). Data comes from upload only.

## 0) Config

In [ ]:
from pathlib import Path

MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 3072
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 2       # set to 1 if CUDA OOM
GRAD_ACCUM = 4
SEED = 42

SAVE_GGUF_TO_DRIVE = True
DRIVE_GGUF_DIR = "/content/drive/MyDrive/Labpulse/models"

WORK = Path("/content/labpulse")
DATA_DIR = WORK / "data"
ADAPTER_DIR = WORK / "adapters"
MERGED_DIR = WORK / "merged_hf"
GGUF_DIR = WORK / "gguf"
GGUF_OUT = GGUF_DIR / "labpulse-1.5b-q4_k_m.gguf"

for p in (WORK, DATA_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("WORK =", WORK)
print("MODEL_ID =", MODEL_ID)
print("Upload target =", DATA_DIR)

## 1) Install dependencies

Run once per new Colab runtime.

In [ ]:
import subprocess
import sys


def pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


# Unsloth only — pulls its own stack; no HF login
pip_install("unsloth")
pip_install("datasets", "trl")

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Runtime → Change runtime type → GPU (T4) → Save, then Runtime → Run all again."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM_GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("CUDA:", torch.version.cuda)
print("torch:", torch.__version__)

## 2) Upload dataset

Upload **one** of these from your Mac:

- Preferred: `Labpulse/data/colab_dataset.zip`
- Or both: `train.jsonl` + `valid.jsonl`

A file picker appears when you run the next cell.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

from google.colab import files

WORK = Path("/content/labpulse")
DATA_DIR = WORK / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "train.jsonl"
valid_path = DATA_DIR / "valid.jsonl"

# Re-run friendly: skip picker if files already present
if train_path.exists() and valid_path.exists() and train_path.stat().st_size > 1000:
    print("Dataset already present — skipping upload.")
    print("train:", train_path, train_path.stat().st_size, "bytes")
    print("valid:", valid_path, valid_path.stat().st_size, "bytes")
else:
    print("Choose colab_dataset.zip  OR  train.jsonl + valid.jsonl")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No files uploaded. Run this cell again and select the dataset files.")

    for name, blob in uploaded.items():
        dest = DATA_DIR / Path(name).name
        dest.write_bytes(blob)
        print("saved", dest, len(blob), "bytes")

        if dest.suffix.lower() == ".zip":
            with zipfile.ZipFile(dest, "r") as zf:
                zf.extractall(DATA_DIR)
            print("extracted zip →", DATA_DIR)

if not train_path.exists():
    raise FileNotFoundError(f"Missing {train_path}. Upload colab_dataset.zip or train.jsonl.")
if not valid_path.exists():
    raise FileNotFoundError(f"Missing {valid_path}. Upload colab_dataset.zip or valid.jsonl.")

print("OK dataset files ready")

In [ ]:
from datasets import Dataset

REQUIRED_TOP_KEYS = {"cover_page", "anomaly_check", "report_body"}


def load_jsonl(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if "messages" not in obj:
                raise ValueError(f"{path.name}:{i} missing messages[]")
            msgs = obj["messages"]
            if len(msgs) < 3:
                raise ValueError(f"{path.name}:{i} need system+user+assistant")
            if msgs[0]["role"] != "system" or msgs[1]["role"] != "user" or msgs[-1]["role"] != "assistant":
                raise ValueError(f"{path.name}:{i} unexpected roles")
            assistant = json.loads(msgs[-1]["content"])
            missing = REQUIRED_TOP_KEYS - set(assistant)
            if missing:
                raise ValueError(f"{path.name}:{i} assistant JSON missing {missing}")
            rows.append({"messages": msgs})
    return rows


train_rows = load_jsonl(train_path)
valid_rows = load_jsonl(valid_path)
print(f"train={len(train_rows)} valid={len(valid_rows)}")

if len(train_rows) < 100:
    raise RuntimeError("train set looks too small — re-upload colab_dataset.zip")
if len(valid_rows) < 10:
    raise RuntimeError("valid set looks too small — re-upload colab_dataset.zip")

print("sample roles:", [m["role"] for m in train_rows[0]["messages"]])
print("assistant keys:", sorted(json.loads(train_rows[0]["messages"][-1]["content"]).keys()))

## 3) Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

## 4) Format chat examples

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}


train_ds = Dataset.from_list(train_rows).map(formatting_prompts_func, batched=True)
valid_ds = Dataset.from_list(valid_rows).map(formatting_prompts_func, batched=True)

preview = train_ds[0]["text"]
print("chars:", len(preview))
print(preview[:400], "...")

## 5) Train LoRA

~30–90 minutes on a free T4. Keep this tab open.

In [ ]:
import inspect

from trl import SFTConfig, SFTTrainer

use_bf16 = torch.cuda.is_bf16_supported()

cfg_kwargs = dict(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=max(1, BATCH_SIZE),
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.03,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    fp16=not use_bf16,
    bf16=use_bf16,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

sft_params = inspect.signature(SFTConfig.__init__).parameters
if "max_seq_length" in sft_params:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
elif "max_length" in sft_params:
    cfg_kwargs["max_length"] = MAX_SEQ_LENGTH

sft_args = SFTConfig(**cfg_kwargs)

trainer_kwargs = dict(
    model=model,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    args=sft_args,
)
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in trainer_params and "dataset_text_field" not in sft_params:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in trainer_params and "max_seq_length" not in cfg_kwargs:
    trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

trainer = SFTTrainer(**trainer_kwargs)
train_result = trainer.train()
print(train_result.metrics)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved adapters →", ADAPTER_DIR)
print("files:", sorted(p.name for p in ADAPTER_DIR.iterdir() if p.is_file())[:12])

## 6) Smoke-test JSON generation

In [ ]:
FastLanguageModel.for_inference(model)

sample_messages = train_rows[0]["messages"][:2]
prompt = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(
    **inputs,
    max_new_tokens=768,
    temperature=0.1,
    do_sample=True,
    use_cache=True,
)
text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(text[:2000])

try:
    start, end = text.find("{"), text.rfind("}")
    parsed = json.loads(text[start : end + 1])
    print("JSON keys:", sorted(parsed.keys()))
except Exception as e:
    print("JSON parse note (safe to continue):", e)

## 7) Export GGUF Q4_K_M (Unsloth only)

Merge + quantize with Unsloth — no Hugging Face login, no PEFT fallback, no llama.cpp.

In [ ]:
import shutil
from pathlib import Path

WORK = Path("/content/labpulse")
ADAPTER_DIR = WORK / "adapters"
MERGED_DIR = WORK / "merged_hf"
GGUF_DIR = WORK / "gguf"
GGUF_OUT = GGUF_DIR / "labpulse-1.5b-q4_k_m.gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

if GGUF_OUT.exists():
    GGUF_OUT.unlink()

gguf_ok = False

# Path A — Unsloth native GGUF from LoRA model
try:
    export_dir = GGUF_DIR / "unsloth_export"
    if export_dir.exists():
        shutil.rmtree(export_dir)
    model.save_pretrained_gguf(
        str(export_dir),
        tokenizer,
        quantization_method="q4_k_m",
    )
    found = list(export_dir.rglob("*.gguf")) or list(GGUF_DIR.rglob("*.gguf"))
    if not found:
        raise FileNotFoundError("Unsloth produced no .gguf")
    found.sort(key=lambda p: ("q4" not in p.name.lower(), -p.stat().st_size))
    shutil.copy2(found[0], GGUF_OUT)
    gguf_ok = GGUF_OUT.exists() and GGUF_OUT.stat().st_size > 1_000_000
    print("GGUF via Unsloth:", found[0])
except Exception as e:
    print("Unsloth GGUF export failed:", e)

# Always try to write merged HF weights (needed for llama.cpp fallback / archival)
if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

merged_ok = False
try:
    model.save_pretrained_merged(
        str(MERGED_DIR),
        tokenizer,
        save_method="merged_16bit",
    )
    merged_ok = (MERGED_DIR / "config.json").exists()
    print("merged HF via Unsloth")
except Exception as e:
    print("Unsloth merge failed, PEFT fallback:", e)
    from peft import PeftModel

    if MERGED_DIR.exists():
        shutil.rmtree(MERGED_DIR)
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    base, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=False,
    )
    peft_model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
    peft_model = peft_model.merge_and_unload()
    peft_model.save_pretrained(str(MERGED_DIR))
    tok.save_pretrained(str(MERGED_DIR))
    merged_ok = (MERGED_DIR / "config.json").exists()

if not merged_ok:
    raise RuntimeError("Could not write merged HF model")

print("merged →", MERGED_DIR)

# Path B — llama.cpp if Unsloth GGUF missing
if not gguf_ok:
    print("Using llama.cpp convert + quantize…")
    FP16_GGUF = GGUF_DIR / "labpulse-1.5b-f16.gguf"
    LLAMA_CPP = Path("/content/llama.cpp")
    if not LLAMA_CPP.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/ggerganov/llama.cpp", str(LLAMA_CPP),
        ])

    pip_install("gguf", "numpy", "sentencepiece", "protobuf")

    convert_script = LLAMA_CPP / "convert_hf_to_gguf.py"
    if not convert_script.exists():
        raise FileNotFoundError(convert_script)

    subprocess.check_call([
        sys.executable,
        str(convert_script),
        str(MERGED_DIR),
        "--outfile", str(FP16_GGUF),
        "--outtype", "f16",
    ])

    quantize_bin = LLAMA_CPP / "build" / "bin" / "llama-quantize"
    if not quantize_bin.exists():
        subprocess.call([
            "bash", "-lc",
            "command -v cmake >/dev/null || apt-get update && apt-get install -y cmake build-essential",
        ])
        subprocess.check_call([
            "cmake", "-S", str(LLAMA_CPP), "-B", str(LLAMA_CPP / "build"),
            "-DGGML_CUDA=OFF",
        ])
        subprocess.check_call([
            "cmake", "--build", str(LLAMA_CPP / "build"), "--config", "Release", "-j",
            "--target", "llama-quantize",
        ])

    subprocess.check_call([
        str(quantize_bin), str(FP16_GGUF), str(GGUF_OUT), "Q4_K_M",
    ])

if not (GGUF_OUT.exists() and GGUF_OUT.stat().st_size > 1_000_000):
    raise RuntimeError(f"GGUF export failed: {GGUF_OUT}")

print("GGUF ready:", GGUF_OUT, "size_mb=", round(GGUF_OUT.stat().st_size / 1e6, 1))

## 8) Download artifacts

Put the GGUF here on your laptop:

```text
Labpulse/models/labpulse-1.5b-q4_k_m.gguf
```

In [ ]:
from google.colab import files, drive

if not GGUF_OUT.exists():
    raise FileNotFoundError(GGUF_OUT)

if SAVE_GGUF_TO_DRIVE:
    drive.mount("/content/drive", force_remount=False)
    dest_dir = Path(DRIVE_GGUF_DIR)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / GGUF_OUT.name
    shutil.copy2(GGUF_OUT, dest)
    print("Drive copy →", dest)

print("Downloading", GGUF_OUT.name)
files.download(str(GGUF_OUT))

adapter_zip = WORK / "labpulse_adapters"
shutil.make_archive(str(adapter_zip), "zip", ADAPTER_DIR)
files.download(str(adapter_zip) + ".zip")

print("Done.")
print("1) Move labpulse-1.5b-q4_k_m.gguf → Labpulse/models/")
print("2) Restart local API without LABPULSE_STUB to use the fine-tuned model")

## Troubleshooting

| Issue | Fix |
|-------|-----|
| CUDA OOM | Config: `BATCH_SIZE = 1`, optionally `MAX_SEQ_LENGTH = 2048`, re-run from model load |
| Upload cell skipped / missing files | Delete `/content/labpulse/data/*.jsonl` then re-run upload cell |
| Browser download stalls | Use Drive copy at `MyDrive/Labpulse/models/` |
| Colab disconnect mid-train | Re-run Install → Config → Upload (skip if present) → Train |